<a href="https://colab.research.google.com/github/vindhyawasan/email-spam-detection/blob/main/email_spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
df = pd.read_csv('/content/drive/MyDrive/enron_spam_data.csv')

In [4]:
df.head()

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [6]:
df.isnull().sum()

,0
Message ID,0
Subject,289
Message,371
Spam/Ham,0
Date,0


In [7]:
df.drop(['Message ID','Date'],axis=1,inplace=True)

In [8]:
df.head()

,Subject,Message,Spam/Ham
0,christmas tree farm pictures,NaN,ham
1,"vastar resources , inc .","gary , production from the high island larger ...",ham
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham
3,re : issue,fyi - see note below - already done .\nstella\...,ham
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham


In [9]:
df.shape

(33716, 3)

In [10]:
df.duplicated().sum()

np.int64(3222)

In [11]:
df.drop_duplicates(inplace=True)

In [12]:
df.shape

(30494, 3)

In [13]:
df.dropna(inplace=True)

In [14]:
df.shape

(30036, 3)

# Removing Unwanted Text and symbol

In [15]:
df["Text"] = df["Subject"] + "" + df["Message"]

In [16]:
df.head()

,Subject,Message,Spam/Ham,Text
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,"vastar resources , inc .gary , production from..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,calpine daily gas nomination- calpine daily ga...
3,re : issue,fyi - see note below - already done .\nstella\...,ham,re : issuefyi - see note below - already done ...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,meter 7268 nov allocationfyi .\n- - - - - - - ...
5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,"mcmullen gas for 11 / 99jackie ,\nsince the in..."


In [17]:
import re

def clean_text(text):
    text = text.lower()                       # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'\S+@\S+', '', text)       # remove email addresses
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # remove symbols/numbers
    text = re.sub(r'\s+', ' ', text)          # remove extra spaces
    text = text.strip()                       # spaces from start/end

    return text

In [18]:
df['Text'] = df['Text'].apply(clean_text)

In [19]:
df.head()

,Subject,Message,Spam/Ham,Text
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,vastar resources inc gary production from the ...
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,calpine daily gas nomination calpine daily gas...
3,re : issue,fyi - see note below - already done .\nstella\...,ham,re issuefyi see note below already done stella...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,meter nov allocationfyi forwarded by lauri a a...
5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,mcmullen gas for jackie since the inlet to riv...


In [20]:
X = df["Text"]
y = df["Spam/Ham"]

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

In [23]:
vectorizer = TfidfVectorizer()

In [24]:
X_train_Tfidf = vectorizer.fit_transform(X_train)
X_test_Tfidf = vectorizer.transform(X_test)

In [25]:
nb_model = MultinomialNB()
nb_model.fit(X_train_Tfidf, y_train)

MultinomialNB()

In [26]:
y_pred = nb_model.predict(X_test_Tfidf)

In [27]:
from sklearn.metrics import accuracy_score, classification_report,f1_score,confusion_matrix, mean_squared_error,mean_absolute_error

In [28]:
acc = accuracy_score(y_test,y_pred)
print(classification_report(y_test, y_pred))
f1 = f1_score(y_test, y_pred, pos_label="spam")

print("F1 Score:", f1)
print("Accuracy:", acc)

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99      3180
        spam       0.99      0.98      0.99      2828

    accuracy                           0.99      6008
   macro avg       0.99      0.99      0.99      6008
weighted avg       0.99      0.99      0.99      6008

F1 Score: 0.9852023533606703
Accuracy: 0.986185086551265


In [29]:
import joblib

joblib.dump(nb_model, "spam_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']